# 04 · Modelo dimensional en Gold: SCD2, hechos y data marts
| Tabla | Tipo | Clave | Cómo se mantiene |
|---|---|---|---|
| `gold.dim_cliente` | SCD Tipo 2 | `customer_sk` (versión) / `customer_id` | cierra la versión vigente y abre otra si cambia segmento o ubicación |
| `gold.dim_producto` | SCD Tipo 1 | `product_card_id` | sobrescribe atributos |
| `gold.dim_ubicacion` | estática | `ubicacion_id` (hash) | solo inserta nuevas |
| `gold.fact_envios` | hechos | `(order_id, order_item_id)` | conserva el `customer_sk` con el que se cargó |

Las dimensiones se alimentan de **todas** las filas válidas del lote (`silver.stg_validos`), porque un cambio de cliente puede llegar en una fila repetida que Silver no reescribe.
Los hechos se alimentan solo de lo que Silver insertó o actualizó en este lote.

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from pyspark.sql import functions as F
from src import config as C
from src.transformaciones import ultimo_por_clave, con_hash_atributos, id_ubicacion, preparar_staging_scd2
from src.sql_merges import merge_dim_cliente_scd2, merge_dim_producto_scd1, merge_dim_ubicacion, merge_fact
from src.sql_marts import todos_los_marts
from src.utilidades import crear_tabla_vacia_si_no_existe, merge_con_registro

dbutils.widgets.text("catalogo", C.CATALOGO_POR_DEFECTO)
dbutils.widgets.text("lote", "lote_01")
cat, lote = dbutils.widgets.get("catalogo"), dbutils.widgets.get("lote")
t = lambda nombre: f"{cat}.{nombre}"

stg = spark.table(t(C.T_STG))
assert stg.filter(F.col("lote_id") != lote).count() == 0, "El staging no corresponde a este lote"

## 1. dim_cliente (SCD Tipo 2)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {t(C.T_DIM_CLIENTE)} (
  customer_sk BIGINT COMMENT 'Clave surrogate de la versión (hash de customer_id + fecha_inicio)',
  customer_id STRING COMMENT 'Clave de negocio',
  customer_segment STRING, customer_city STRING, customer_state STRING, customer_country STRING,
  hash_atributos STRING COMMENT 'Huella de los atributos versionados',
  fecha_inicio DATE COMMENT 'Inicio de vigencia (incluido)',
  fecha_fin DATE COMMENT 'Fin de vigencia (excluido); 9999-12-31 si es la versión actual',
  es_vigente BOOLEAN
) COMMENT 'Dimensión de clientes con historial (SCD Tipo 2)'
""")

vigentes = spark.table(t(C.T_DIM_CLIENTE)).filter("es_vigente = true").select("customer_id", "hash_atributos")
staging_cli = preparar_staging_scd2(stg, vigentes, "customer_id", C.ATRIBUTOS_CLIENTE)
staging_cli.createOrReplaceTempView("v_stg_cliente")
print("Staging SCD2 → altas de versión:", staging_cli.filter("merge_key IS NULL").count(),
      "| versiones a cerrar:", staging_cli.filter("merge_key IS NOT NULL").count())

print(merge_con_registro(spark, merge_dim_cliente_scd2(t(C.T_DIM_CLIENTE), "v_stg_cliente"),
                         cat, C.T_DIM_CLIENTE, lote, C.T_LOG))

## 2. dim_producto (SCD1) y dim_ubicacion

In [0]:
prod = con_hash_atributos(ultimo_por_clave(stg, "product_card_id"), C.ATRIBUTOS_PRODUCTO) \
    .select("product_card_id", *C.ATRIBUTOS_PRODUCTO, "hash_atributos")
crear_tabla_vacia_si_no_existe(spark, t(C.T_DIM_PRODUCTO), prod)
prod.createOrReplaceTempView("v_stg_producto")
print(merge_con_registro(spark, merge_dim_producto_scd1(t(C.T_DIM_PRODUCTO), "v_stg_producto"),
                         cat, C.T_DIM_PRODUCTO, lote, C.T_LOG))

ubi = stg.select(*C.ATRIBUTOS_UBICACION).distinct().withColumn("ubicacion_id", id_ubicacion()) \
    .select("ubicacion_id", *C.ATRIBUTOS_UBICACION)
crear_tabla_vacia_si_no_existe(spark, t(C.T_DIM_UBICACION), ubi)
ubi.createOrReplaceTempView("v_stg_ubicacion")
print(merge_con_registro(spark, merge_dim_ubicacion(t(C.T_DIM_UBICACION), "v_stg_ubicacion"),
                         cat, C.T_DIM_UBICACION, lote, C.T_LOG))

## 3. fact_envios

In [0]:
silver_lote = spark.table(t(C.T_SILVER)).filter(F.col("lote_id") == lote)   # filas nuevas o cambiadas en este lote
cli_vig = spark.table(t(C.T_DIM_CLIENTE)).filter("es_vigente = true").select("customer_id", "customer_sk")

fact = (silver_lote.join(cli_vig, "customer_id", "left")
        .withColumn("ubicacion_id", id_ubicacion())
        .withColumn("es_tardio", F.col("days_for_shipping_real") > F.col("days_for_shipment_scheduled"))
        .select(*C.COLUMNAS_FACT))
crear_tabla_vacia_si_no_existe(spark, t(C.T_FACT), fact)
fact.createOrReplaceTempView("v_stg_fact")
print(f"Filas candidatas a hechos en este lote: {fact.count()}")
print(merge_con_registro(spark, merge_fact(t(C.T_FACT), "v_stg_fact"), cat, C.T_FACT, lote, C.T_LOG))

## 4. Data marts (3)

In [0]:
for sql in todos_los_marts(cat):
    spark.sql(sql)
for m in (C.T_MART_ENTREGAS, C.T_MART_VENTAS, C.T_MART_RETRASO):
    print(m, "→", spark.table(t(m)).count(), "filas")

## 5. Metadatos descriptivos (comentarios de tabla)

In [0]:
comentarios = {
    C.T_BRONZE: "Entrada tal cual (texto) + lote_id/fecha_ingesta/archivo_origen. Sin datos personales.",
    C.T_SILVER: "Envíos validados y deduplicados por (order_id, order_item_id). Última versión conocida de cada línea.",
    C.T_RECHAZOS: "Filas que incumplen alguna regla de calidad, con el motivo.",
    C.T_METRICAS: "Métricas de calidad por lote y regla.",
    C.T_LOG: "Control de cargas: filas insertadas/actualizadas por MERGE, tabla y lote.",
    C.T_FACT: "Hechos de envío a nivel de línea de pedido. customer_sk apunta a la versión de cliente vigente al cargar la fila.",
}
for tabla, texto in comentarios.items():
    spark.sql(f"COMMENT ON TABLE {t(tabla)} IS '{texto}'")